# Маршрутизация заявок поддержки
По умолчанию загружается CSV из 12 000 СИНТЕТИЧЕСКИХ заявок. Сначала выполните data_checks.ipynb. Синтетические метрики описывают работу на шаблонах. Приёмка по реальным обращениям ожидает исторических данных.

In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from triage import experiment, read_csv, validate, TASKS, is_synthetic

DATA_CSV = Path("data/synthetic/tickets_synthetic.csv")
TEST_IDS_CSV = None  # для существующего теста Path("data/test_ids.csv")
frame = read_csv(DATA_CSV)
demo = is_synthetic(frame)
mode = "SYNTHETIC" if demo else "historical"
out = Path("runs") / mode / uuid4().hex
ids = None if TEST_IDS_CSV is None else read_csv(TEST_IDS_CSV)["id"]
print(mode, "· записей", len(frame), "· каталог", out)
display(validate(frame)[["subject", "text"]].eq("").sum().rename("Пустых полей"))


## Обучение и оценка
Два значения C сравниваются на validation. После фиксации обоих классификаторов каждый baseline оценивается на одном и том же тесте. См. `triage.py` для полного порядка операций.

In [ ]:
metrics = experiment(frame, out, demo=demo, test_ids=ids)
print(mode, "· распределение в train")
display(pd.read_csv(out / "train_counts.csv", index_col=0))
print("Validation")
display(pd.read_csv(out / "validation.csv"))
print(mode, "· test")
display(metrics.round(3))
display(json.loads((out / "run.json").read_text()))

## Матрицы ошибок
Строка — истинная метка, столбец — прогноз. Число в ячейке показывает количество заявок.

In [ ]:
for task in TASKS:
    for method in ["majority", "tfidf_lr"]:
        matrix = pd.read_csv(out / f"matrix_{method}_{task}.csv", index_col=0)
        print(mode, task, method)
        display(matrix)
        if len(matrix) <= 20:
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.imshow(matrix, cmap="Blues")
            ax.set_xticks(range(len(matrix)), matrix.columns, rotation=40, ha="right")
            ax.set_yticks(range(len(matrix)), matrix.index)
            ax.set(xlabel="Прогноз", ylabel="Истинный класс", title=f"{mode} · {method}")
            plt.tight_layout()
            plt.show()

## Пропуски критического приоритета
Сколько критических обращений понижено и до какого уровня? Для содержательного разбора ошибок сопоставьте ID из локального отчёта с обезличенным источником. Проверьте признаки срочности, полноту описания и согласованность разметки. Изменения по итогам разбора потребуют нового независимого теста.

In [ ]:
for method in ["majority", "tfidf_lr"]:
    misses = pd.read_csv(out / f"missed_{method}.csv", dtype=str)
    print(mode, method, "· критических пропущено:", len(misses))
    display(misses.groupby(["predicted", "department"]).size().rename("Количество").to_frame())